In [ ]:
# Run this cell if you're using from colab
#!git clone https://github.com/R-Oc-A/HackathonPastryLPV.git
#!wget https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/IntensityGrids/grids.tar.gz
#!tar -xvf grids.tar.gz
#!pip install https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/wheel/pastrypy-010-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
#!pip install tomli_w
#!pip install pyvista
#!pip install trame-pyvista
#import sys
#sys.path.append('/content/HackathonPastryLPV')
#import os
#os.environ["GRIDS"]="/content/ema_parquets/"

In [ ]:
import pastrypy as psp
import pulsation_description as plsd
import line_profile_description as lpd
import tomli_w
import os
import polars as pl
import matplotlib.pyplot as plt
import pyvista as pv
import numpy as np
from trame_pyvista.jupyter import launch_server

# Pulstar configuration
Here you specify the star you'll be modelling as well as the modes of pulsation

In [ ]:
#Taken from a Simbad quick Query and from Teltings paper
mode=plsd.Mode(l=2,m=1,
                rel_dr=0.0024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = "PerturbativeCoriolis")
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=3))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)

# Profile configuration
Here you specify the line profile variability you want to observe.

In [ ]:
#Taken from a Simbad quick Query
wl_range=lpd.WavelengthRange(start=4551.0,end=4555.0,step=0.0033)
#path_to_grids="../profile/grids/"
path_to_grids = os.getenv("GRIDS")
#path_to_grids = f"{os.getenv("GRIDS")}ema_parquets/"
#print(path_to_grids)
grid1=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_20000_0350_0020.parquet"),Nadya=None)
grid2=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_20000_0380_0020.parquet"),Nadya=None)
grid3=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_24000_0350_0020.parquet"),Nadya=None)
grid4=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_24000_0380_0020.parquet"),Nadya=None)
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)
print(prof_toml_string)

# First run

In [ ]:
pulse_df = psp.pulstar(puls_toml_string)

In [ ]:
pulse_df.sort("area").tail(5)

In [ ]:
wavelength_df = psp.profile(prof_toml_string,pulse_df)

In [ ]:
wavelength_df.head(5)

In [ ]:
#Taken from a Simbad quick Query and from Teltings paper
nonrot= plsd.NonRot()
pertcor=plsd.PerturbCor()
tar = plsd.TAR()
cendef = plsd.CenDef(coefficient_expansion=[-0.856,0.01,0.0])
rotation_regime = plsd.RotationRegime(NonRotating=None,PerturbativeCoriolis=None,Tar=None,CentrifugalDeformation=None)
#rotation_regime.NonRotating=nonrot
#rotation_regime.PerturbativeCoriolis=pertcor
#rotation_regime.Tar=tar
rotation_regime.CentrifugalDeformation=cendef

mode=plsd.Mode(l=2,m=1,
                rel_dr=0.0024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = rotation_regime)
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=3))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)
print(puls_toml_string)

# Testing grid generation. 

In [12]:
sphere_points = psp.sphere_triangulation_points()
sphere_points.head()

I'm here, found some bad points, I guess.
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 4
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 3
length of actual front polygon is 4
--------------------
putting last triangle
front polygons to be processed are 2
length of actual front polygon is 5
--------------------
I'm here, found some bad points, I guess.
I'm here, found some bad points, I guess.
putting last triangle
front polygons to be processed are 1
length of actual front polygon is 4
--------------------
putting last triangle
front polygons to be processed are 0
----------------------------------------
---------Triangulation Finished---------
----------------------------------------


point_id,x coordinate,y coordinate,z coordinate
u32,f64,f64,f64
0,-0.999201,0.0,-0.039968
1,-0.945576,0.0,-0.325401
2,-0.951319,-0.248851,-0.181842
3,-0.962803,-0.248851,0.105277
4,-0.968546,0.0,0.248836


In [13]:
sphere_triangles = psp.sphere_triangulation_triangles()
sphere_triangles.head()

I'm here, found some bad points, I guess.
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 4
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 3
length of actual front polygon is 4
--------------------
putting last triangle
front polygons to be processed are 2
length of actual front polygon is 5
--------------------
I'm here, found some bad points, I guess.
I'm here, found some bad points, I guess.
putting last triangle
front polygons to be processed are 1
length of actual front polygon is 4
--------------------
putting last triangle
front polygons to be processed are 0
----------------------------------------
---------Triangulation Finished---------
----------------------------------------


first vertex,second vertex,third vertex
u32,u32,u32
0,1,2
0,2,3
0,3,4
0,4,5
0,5,6


In [14]:
points_extracted = sphere_points[['x coordinate','y coordinate','z coordinate']].to_numpy()
triangles_extracted = sphere_triangles[['first vertex','second vertex','third vertex']].to_numpy()
faces = np.hstack([np.full((triangles_extracted.shape[0],1),3),triangles_extracted])
mesh = pv.PolyData(points_extracted,faces)

In [17]:
from trame_pyvista.jupyter import launch_server
await launch_server().ready

True

In [18]:
mesh.plot(
    show_edges=True,
    color='green',
    style='wireframe'
)

Widget(value='<iframe src="http://localhost:42639/index.html?ui=P_0x713a6819e0d0_1&reconnect=auto" class="pyvi…

In [19]:
roche_points = psp.roche_deformed_triangulation_points(w=0.4,length=0.3)
roche_triangles = psp.roche_deformed_triangulation_triangles(w=0.4,length=0.3)

breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 5
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 4
length of actual front polygon is 4
--------------------
putting last triangle
front polygons to be processed are 3
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 2
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 1
length of actual front polygon is 4
--------------------
breaking while finishing points with small front angles
putting last triangle
front polygons to be processed are 0
----------------------------------------
---------Triangulation Finished---------

In [20]:
points_extracted = roche_points[['x coordinate','y coordinate','z coordinate']].to_numpy()
triangles_extracted = roche_triangles[['first vertex','second vertex','third vertex']].to_numpy()
faces = np.hstack([np.full((triangles_extracted.shape[0],1),3),triangles_extracted])
mesh = pv.PolyData(points_extracted,faces)

In [21]:
plotter = pv.Plotter()
actor = plotter.add_mesh(mesh,color='magenta',show_edges=True)
plotter.reset_camera()
plotter.set_position([0,4,-1])
plotter.set_viewup([0,-1,-1])
plotter.show()

Widget(value='<iframe src="http://localhost:42639/index.html?ui=P_0x713ab65d9590_2&reconnect=auto" class="pyvi…